# Three-Layer GCN on Cora

This notebook trains a vanilla three-layer graph convolutional network for semi-supervised node classification on the Cora citation network.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

torch.manual_seed(42)

dataset = Planetoid(root="data/Planetoid", name="Cora")
data = dataset[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = data.to(device)
print(f"Nodes: {data.num_nodes}, Features: {dataset.num_node_features}, Classes: {dataset.num_classes}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv3(x, edge_index)


model = GCN(dataset.num_node_features, 16, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
def train():
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index)
    loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


def classification_metrics(predictions, labels, num_classes):
    """Return macro precision, recall, F1, and the confusion matrix."""
    confusion = torch.zeros(
        (num_classes, num_classes), dtype=torch.long, device=labels.device
    )
    indices = labels * num_classes + predictions
    confusion.view(-1).scatter_add_(
        0, indices, torch.ones_like(indices, dtype=torch.long)
    )

    true_positive = confusion.diag().float()
    predicted = confusion.sum(dim=0).float()
    actual = confusion.sum(dim=1).float()
    precision_per_class = torch.nan_to_num(true_positive / predicted)
    recall_per_class = torch.nan_to_num(true_positive / actual)
    f1_per_class = torch.nan_to_num(
        2 * precision_per_class * recall_per_class
        / (precision_per_class + recall_per_class)
    )
    return (
        precision_per_class.mean().item(),
        recall_per_class.mean().item(),
        f1_per_class.mean().item(),
        confusion.cpu(),
    )


@torch.no_grad()
def test():
    model.eval()
    logits = model(data.x, data.edge_index)
    predictions = logits.argmax(dim=1)
    results = []
    for mask in (data.train_mask, data.val_mask, data.test_mask):
        labels = data.y[mask]
        masked_predictions = predictions[mask]
        accuracy = (masked_predictions == labels).float().mean().item()
        precision, recall, f1, confusion = classification_metrics(
            masked_predictions, labels, dataset.num_classes
        )
        results.append({
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "confusion": confusion,
        })
    return results


for epoch in range(1, 201):
    loss = train()
    if epoch == 1 or epoch % 20 == 0:
        train_metrics, val_metrics, test_metrics = test()
        print(
            f"Epoch {epoch:03d} | Loss {loss:.4f} | "
            f"Train accuracy {train_metrics['accuracy']:.4f} | "
            f"Val accuracy {val_metrics['accuracy']:.4f} | "
            f"Test accuracy {test_metrics['accuracy']:.4f}"
        )

train_metrics, val_metrics, test_metrics = test()
for split_name, metrics in (
    ("Train", train_metrics),
    ("Validation", val_metrics),
    ("Test", test_metrics),
):
    print(
        f"{split_name}: accuracy={metrics['accuracy']:.4f}, "
        f"precision_macro={metrics['precision']:.4f}, "
        f"recall_macro={metrics['recall']:.4f}, "
        f"f1_macro={metrics['f1']:.4f}"
    )
    print(f"{split_name} confusion matrix:\n{metrics['confusion']}")